# SAOU Chiral-Wave Sanity Check

This notebook makes the chiral motion in the shell-resolved antisymmetric Ornstein-Uhlenbeck field visible in real space.  The main movie colors each site by the internal angle `theta = atan2(v, u)` and overlays direction arrows.  No wavelet transform is used: the visualization is a direct real-space view of the simulated field.

The coherent wave below is produced by a single-mode initial condition and then allowed to evolve freely.  This is a visualization sanity check, not a central-spin forcing experiment.

In [ ]:
import os
import sys

candidate_roots = [
    os.environ.get("CNEEP_V2_ROOT"),
    os.path.abspath("../.."),
    os.path.abspath("../../.."),
    os.path.abspath(".."),
    os.path.abspath("."),
]

CNEEP_V2_ROOT = None
for candidate in candidate_roots:
    if candidate and os.path.exists(os.path.join(candidate, "data", "SAOU", "saou_model.py")):
        CNEEP_V2_ROOT = candidate
        break

if CNEEP_V2_ROOT is None:
    raise RuntimeError("Could not locate CNEEP_v2 root. Set CNEEP_V2_ROOT.")

SAOU_DIR = os.path.join(CNEEP_V2_ROOT, "data", "SAOU")
for path in (CNEEP_V2_ROOT, SAOU_DIR):
    if path not in sys.path:
        sys.path.append(path)

print("CNEEP_v2 root:", CNEEP_V2_ROOT)

In [ ]:
import math
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import animation as mpl_animation
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

from saou_model import (
    build_shell_ops,
    drift,
    irreversible_velocity_total,
    make_annular_shells,
    simulate,
    theoretical_epr_gram,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["animation.embed_limit"] = 250
print("device:", device)

## 1. Parameters and predicted seeded-mode rotation

For a single complex mode `z = u + i v ~ exp(i k dot r)`, the linear SAOU dynamics rotates the phase at a mode-dependent angular frequency.  We use that fact only to choose a mode that moves visibly; the movie itself is drawn directly from the simulated trajectory.

In [ ]:
# Visual sanity-check parameters.
L = 64
mode_y = 0
mode_x = 3
seed = 4

radii = (1, 2, 4, 8)
amplitudes = (1.0, 0.8, 1.2, 0.5)
gamma = 0.08
omega0 = 0.0
T = 0.02
dt = 2.0e-3
n_steps = 6_000
burn_steps = 0
sample_every = 10
weight_normalization = "mean"

def mode_frequency(L, radii, amplitudes, mode_y, mode_x, omega0=0.0, weight_normalization="mean"):
    shells = make_annular_shells(
        radii,
        amplitudes,
        weight_normalization=weight_normalization,
    )
    ky = 2.0 * np.pi * mode_y / L
    kx = 2.0 * np.pi * mode_x / L
    omega = float(omega0)
    rows = []
    for sh in shells:
        lam = 0.0
        for (dy, dx), w in zip(sh.offsets, sh.weights):
            lam += float(w) * np.cos(kx * dx + ky * dy)
        c = float(np.sum(sh.weights))
        contribution = float(sh.amplitude) * (lam - c)
        omega += contribution
        rows.append((sh.name, float(sh.amplitude), lam - c, contribution))
    return omega, rows

omega_mode, omega_rows = mode_frequency(
    L, radii, amplitudes, mode_y, mode_x, omega0, weight_normalization
)
period = 2.0 * np.pi / abs(omega_mode) if abs(omega_mode) > 1e-12 else np.inf
total_time = n_steps * dt

print(f"L={L}, seeded mode=(ky={mode_y}, kx={mode_x})")
print(f"predicted angular frequency: {omega_mode:.6f}")
print(f"predicted period:            {period:.3f}")
print(f"simulation time:             {total_time:.3f}")
print()
for name, amp, lap, contribution in omega_rows:
    print(f"{name:14s} amp={amp: .3f}  lambda-c={lap: .6f}  contribution={contribution: .6f}")

## 2. Single-mode initial condition

A random stationary SAOU snapshot has no fixed spatial phase, so an angle movie usually looks noisy.  Here we seed one clean mode once at `t=0`; after that the dynamics is unforced.

In [ ]:
rng = np.random.default_rng(seed)
yy, xx = np.meshgrid(np.arange(L), np.arange(L), indexing="ij")
spatial_phase = 2.0 * np.pi * (mode_y * yy + mode_x * xx) / L

initial_amplitude = 1.5
noise_amplitude = 0.03
x0 = np.empty((L, L, 2), dtype=np.float64)
x0[..., 0] = initial_amplitude * np.cos(spatial_phase)
x0[..., 1] = initial_amplitude * np.sin(spatial_phase)
x0 += noise_amplitude * rng.standard_normal(size=x0.shape)

theta0 = np.arctan2(x0[..., 1], x0[..., 0])
mag0 = np.sqrt(np.sum(x0**2, axis=-1))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(theta0, origin="lower", cmap="twilight", vmin=-np.pi, vmax=np.pi)
axes[0].set_title("initial angle theta")
plt.colorbar(im0, ax=axes[0], fraction=0.046, label="radians")

im1 = axes[1].imshow(mag0, origin="lower", cmap="magma")
axes[1].set_title("initial magnitude |z|")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## 3. Simulate free SAOU relaxation

In [ ]:
out = simulate(
    L=L,
    M=1,
    device=device,
    radii=radii,
    amplitudes=amplitudes,
    gamma=gamma,
    omega0=omega0,
    T=T,
    dt=dt,
    n_steps=n_steps,
    burn_steps=burn_steps,
    sample_every=sample_every,
    seed=seed,
    record_trajectory=True,
    weight_normalization=weight_normalization,
    x0=x0,
    show_progress=True,
)

traj = out["trajectory"]
dt_frame = dt * sample_every
time = out["params"]["dt"] * (1 + out["params"]["sample_every"] * np.arange(len(traj)))
z = traj[..., 0] + 1j * traj[..., 1]
theta = np.angle(z)
mag = np.abs(z)

print("trajectory:", traj.shape)
print("dt_frame:", dt_frame)
print("saved frames:", len(traj))

## 4. Angle-color snapshots

A cyclic colormap is essential here.  Ordinary red-blue maps create a fake jump at `theta = pi`, while `twilight` treats angle as periodic.

In [ ]:
snap_ids = [0, len(traj) // 3, 2 * len(traj) // 3, len(traj) - 1]
fig, axes = plt.subplots(2, len(snap_ids), figsize=(3.4 * len(snap_ids), 6.2))

for col, idx in enumerate(snap_ids):
    im = axes[0, col].imshow(theta[idx], origin="lower", cmap="twilight", vmin=-np.pi, vmax=np.pi)
    axes[0, col].set_title(f"theta, t={time[idx]:.2f}")
    axes[0, col].set_xticks([])
    axes[0, col].set_yticks([])

    im_mag = axes[1, col].imshow(mag[idx], origin="lower", cmap="magma")
    axes[1, col].set_title("|z|")
    axes[1, col].set_xticks([])
    axes[1, col].set_yticks([])

fig.colorbar(im, ax=axes[0, :].ravel().tolist(), fraction=0.025, pad=0.02, label="theta")
fig.colorbar(im_mag, ax=axes[1, :].ravel().tolist(), fraction=0.025, pad=0.02, label="|z|")
plt.show()

## 5. Angle-color movie with direction arrows

In [ ]:
max_movie_frames = 180
fps = 24
arrow_stride = 4
output_dir = Path(CNEEP_V2_ROOT) / "results" / "saou_chiral_sanity"
output_dir.mkdir(parents=True, exist_ok=True)

frame_ids = np.linspace(0, len(traj) - 1, min(max_movie_frames, len(traj)), dtype=int)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), dpi=120)
angle_im = axes[0].imshow(theta[frame_ids[0]], origin="lower", cmap="twilight", vmin=-np.pi, vmax=np.pi)
mag_vmax = max(1e-12, float(np.percentile(mag, 99)))
mag_im = axes[1].imshow(mag[frame_ids[0]], origin="lower", cmap="magma", vmin=0, vmax=mag_vmax)

ys, xs = np.mgrid[0:L:arrow_stride, 0:L:arrow_stride]
initial_angle = theta[frame_ids[0], ::arrow_stride, ::arrow_stride]
quiv = axes[0].quiver(
    xs + 0.5,
    ys + 0.5,
    np.cos(initial_angle),
    np.sin(initial_angle),
    color="black",
    pivot="middle",
    scale=32,
    width=0.003,
    alpha=0.72,
)

axes[0].set_title("angle theta = atan2(v, u)")
axes[1].set_title("magnitude |z|")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(angle_im, ax=axes[0], fraction=0.046, label="theta")
fig.colorbar(mag_im, ax=axes[1], fraction=0.046, label="|z|")
title = fig.suptitle("")

def update_movie(i):
    idx = int(frame_ids[i])
    ang = theta[idx]
    angle_im.set_data(ang)
    mag_im.set_data(mag[idx])
    ang_sub = ang[::arrow_stride, ::arrow_stride]
    quiv.set_UVC(np.cos(ang_sub), np.sin(ang_sub))
    title.set_text(f"SAOU chiral-wave sanity | frame={idx}, t={time[idx]:.2f}")
    return angle_im, mag_im, quiv, title

anim = FuncAnimation(fig, update_movie, frames=len(frame_ids), interval=1000 / fps, blit=False)

saved_paths = []
save_errors = []
mp4_path = output_dir / "saou_chiral_angle_movie.mp4"
gif_path = output_dir / "saou_chiral_angle_movie.gif"

try:
    if mpl_animation.writers.is_available("ffmpeg"):
        anim.save(str(mp4_path), writer=mpl_animation.FFMpegWriter(fps=fps, bitrate=2400), dpi=120)
        saved_paths.append(mp4_path)
    elif mpl_animation.writers.is_available("pillow"):
        anim.save(str(gif_path), writer=mpl_animation.PillowWriter(fps=fps), dpi=100)
        saved_paths.append(gif_path)
    else:
        save_errors.append("No ffmpeg or pillow writer found; displaying embedded JS animation only.")
except Exception as exc:
    save_errors.append(str(exc))

html = anim.to_jshtml(fps=fps)
plt.close(fig)

if saved_paths:
    links = "<br>".join(f'<a href="{p.resolve().as_uri()}">{p.name}</a>' for p in saved_paths)
    display(HTML(f"<b>Saved animation:</b><br>{links}"))
if save_errors:
    display(HTML("<br>".join(f"<small>{err}</small>" for err in save_errors)))
display(HTML(html))

## 6. Space-time phase cut

This is still a real-space diagnostic: take one row of the angle-colored field and stack it over time.  Diagonal bands mean the seeded phase profile is moving.

In [ ]:
row = L // 2
phase_xt = theta[:, row, :]

fig, ax = plt.subplots(figsize=(10, 4.5))
im = ax.imshow(
    phase_xt,
    aspect="auto",
    origin="lower",
    cmap="twilight",
    vmin=-np.pi,
    vmax=np.pi,
    extent=[0, L, time[0], time[-1]],
)
ax.set_xlabel("x")
ax.set_ylabel("time")
ax.set_title(f"space-time angle cut at y={row}")
fig.colorbar(im, ax=ax, label="theta")
plt.tight_layout()
plt.show()

## 7. Instantaneous signed EP-density map

The total SAOU stochastic EP increment density is estimated as `v_irr(x_mid) dot dx / T`.  A single-frame map is signed and noisy, but it lets you check whether the EP signal has spatial structure in the same frames where the angle wave is visible.

In [ ]:
shells = make_annular_shells(radii, amplitudes, weight_normalization=weight_normalization)
ops = build_shell_ops(L, shells)
torch_device = torch.device(device)
for op in ops:
    op.kernel_hat_t = torch.from_numpy(op.kernel_hat).to(device=torch_device, dtype=torch.complex128)

def ep_density_between(traj, idx):
    x_start = traj[idx]
    x_end = traj[idx + 1]
    x_mid = 0.5 * (x_start + x_end)
    dx_field = x_end - x_start
    with torch.no_grad():
        x_mid_t = torch.from_numpy(x_mid).to(device=torch_device, dtype=torch.float64)
        vel = irreversible_velocity_total(x_mid_t, ops, omega0=omega0).cpu().numpy()
    ep_map = np.sum(vel * dx_field, axis=-1) / (T * dt_frame)
    angular_flux = (
        x_mid[..., 0] * dx_field[..., 1] - x_mid[..., 1] * dx_field[..., 0]
    ) / dt_frame
    return ep_map, angular_flux

idx = len(traj) // 3
ep_map, angular_flux = ep_density_between(traj, idx)
vmax_ep = max(1e-12, float(np.percentile(np.abs(ep_map), 99)))
vmax_flux = max(1e-12, float(np.percentile(np.abs(angular_flux), 99)))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
im0 = axes[0].imshow(theta[idx], origin="lower", cmap="twilight", vmin=-np.pi, vmax=np.pi)
axes[0].set_title(f"theta, t={time[idx]:.2f}")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(ep_map, origin="lower", cmap="RdBu_r", vmin=-vmax_ep, vmax=vmax_ep)
axes[1].set_title("signed EP density rate")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(angular_flux, origin="lower", cmap="RdBu_r", vmin=-vmax_flux, vmax=vmax_flux)
axes[2].set_title("angular flux")
fig.colorbar(im2, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

print(f"mean EP density rate at this frame: {ep_map.mean():.6g}")
print(f"spatial std EP density rate:        {ep_map.std():.6g}")

## 8. Maintained wave by weak plane-wave drive

A stable OU process cannot maintain a coherent wave forever without either conditioning/alignment or external energy injection.  Here we add a weak global travelling-wave drive

`dX_i = F_SAOU(X)_i dt + f_i(t) dt + sqrt(2T) dW_i`

with

`f_i(t) = A_drive [cos(k dot r_i + Omega_k t), sin(k dot r_i + Omega_k t)]`.

`Omega_k` is chosen from the autonomous SAOU dispersion for the selected mode, so the drive is nearly static in the co-rotating frame.  Unlike a central spin, this does not create a point defect; it phase-locks one spatial Fourier mode and gives a periodic driven steady state.  Because this is externally driven, the autonomous SAOU EPR theory must be kept separate from the drive-work contribution.

In [ ]:
RUN_DRIVEN_WAVE = True

if RUN_DRIVEN_WAVE:
    try:
        from tqdm.auto import trange
    except ImportError:
        trange = range

    drive_gamma = 0.12
    drive_T = 0.004
    drive_target_amplitude = 1.35
    drive_amp = drive_gamma * drive_target_amplitude
    drive_dt = dt
    # These are intentionally longer than the free-relaxation movie so that the
    # driven mode has time to settle before we judge the pattern.
    drive_burn_steps = 12_000
    drive_n_steps = 24_000
    drive_sample_every = 20
    drive_dt_frame = drive_dt * drive_sample_every

    drive_shells = make_annular_shells(
        radii,
        amplitudes,
        weight_normalization=weight_normalization,
    )
    drive_ops = build_shell_ops(L, drive_shells)
    torch_device = torch.device(device)
    for op in drive_ops:
        op.kernel_hat_t = torch.from_numpy(op.kernel_hat).to(device=torch_device, dtype=torch.complex128)

    spatial_phase_t = torch.from_numpy(spatial_phase).to(device=torch_device, dtype=torch.float64)
    torch.manual_seed(seed + 20)
    x = 0.05 * torch.randn((L, L, 2), device=torch_device, dtype=torch.float64)
    noise_scale = math.sqrt(2.0 * drive_T * drive_dt)

    def plane_wave_drive(t):
        phase_t = spatial_phase_t + omega_mode * float(t)
        force = torch.empty_like(x)
        force[..., 0] = drive_amp * torch.cos(phase_t)
        force[..., 1] = drive_amp * torch.sin(phase_t)
        return force

    with torch.no_grad():
        for step in trange(drive_burn_steps, desc="driven burn-in", leave=False):
            t = step * drive_dt
            force = plane_wave_drive(t)
            x = x + (drift(x, drive_ops, drive_gamma, omega0=omega0) + force) * drive_dt
            x = x + noise_scale * torch.randn_like(x)

        driven_samples = []
        driven_sample_times_abs = []
        for step in trange(drive_n_steps, desc="driven simulation", leave=False):
            t = (drive_burn_steps + step) * drive_dt
            force = plane_wave_drive(t)
            x = x + (drift(x, drive_ops, drive_gamma, omega0=omega0) + force) * drive_dt
            x = x + noise_scale * torch.randn_like(x)

            if step % drive_sample_every == 0:
                driven_samples.append(x.cpu().numpy())
                driven_sample_times_abs.append((drive_burn_steps + step + 1) * drive_dt)

    driven_traj = np.asarray(driven_samples)
    driven_time_abs = np.asarray(driven_sample_times_abs)
    driven_time = driven_time_abs - drive_burn_steps * drive_dt
    driven_z = driven_traj[..., 0] + 1j * driven_traj[..., 1]
    driven_theta = np.angle(driven_z)
    driven_mag = np.abs(driven_z)

    print("driven trajectory:", driven_traj.shape)
    print(f"burn-in time:     {drive_burn_steps * drive_dt:.3f}")
    print(f"production time:  {drive_n_steps * drive_dt:.3f}")
    print(f"saved dt:         {drive_dt_frame:.4f}")
    print(f"drive amplitude: {drive_amp:.4f}")
    print(f"drive frequency: {omega_mode:.6f}")
    print(f"drive target response amplitude near resonance: {drive_amp / drive_gamma:.3f}")
else:
    print("Driven wave skipped.")

In [ ]:
if RUN_DRIVEN_WAVE:
    rotating_basis = np.exp(
        -1j * (spatial_phase[None, ...] + omega_mode * driven_time_abs[:, None, None])
    )
    driven_mode_order = np.mean(driven_z * rotating_basis, axis=(1, 2))
    driven_mode_amp = np.abs(driven_mode_order)
    driven_phase_error = np.unwrap(np.angle(driven_mode_order))
    driven_rms_mag = np.sqrt(np.mean(driven_mag**2, axis=(1, 2)))

    def running_mean_1d(x, window):
        window = max(1, min(int(window), len(x)))
        kernel = np.ones(window, dtype=np.float64) / window
        return np.convolve(x, kernel, mode="same")

    def plateau_score(x):
        q = max(3, len(x) // 4)
        mid = x[-2 * q : -q]
        late = x[-q:]
        scale = np.std(np.concatenate([mid, late])) + 1e-12
        return mid.mean(), late.mean(), abs(late.mean() - mid.mean()) / scale

    amp_mid, amp_late, amp_score = plateau_score(driven_mode_amp)
    rms_mid, rms_late, rms_score = plateau_score(driven_rms_mag)

    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    smooth_window = max(5, len(driven_time) // 40)

    axes[0].plot(driven_time, driven_mode_amp, alpha=0.35, lw=0.8)
    axes[0].plot(driven_time, running_mean_1d(driven_mode_amp, smooth_window), lw=2)
    axes[0].axhline(drive_target_amplitude, color="k", linestyle="--", lw=1, label="target")
    axes[0].set_ylabel("|mode order|")
    axes[0].legend()

    axes[1].plot(driven_time, driven_phase_error, lw=1)
    axes[1].set_ylabel("phase error")

    axes[2].plot(driven_time, driven_rms_mag, alpha=0.35, lw=0.8)
    axes[2].plot(driven_time, running_mean_1d(driven_rms_mag, smooth_window), lw=2)
    axes[2].set_ylabel("RMS |z|")
    axes[2].set_xlabel("time after burn-in")

    fig.suptitle("Driven-mode steady diagnostic")
    plt.tight_layout()
    plt.show()

    print("Plateau check: compare previous quarter vs last quarter")
    print(f"  mode amplitude: middle={amp_mid:.5f}, late={amp_late:.5f}, score={amp_score:.3f}")
    print(f"  RMS magnitude:   middle={rms_mid:.5f}, late={rms_late:.5f}, score={rms_score:.3f}")
    print("A small score means the plotted observable is close to a plateau over this run.")

In [ ]:
if RUN_DRIVEN_WAVE:
    snap_ids = [0, len(driven_traj) // 3, 2 * len(driven_traj) // 3, len(driven_traj) - 1]
    fig, axes = plt.subplots(2, len(snap_ids), figsize=(3.4 * len(snap_ids), 6.2))

    for col, idx in enumerate(snap_ids):
        im = axes[0, col].imshow(
            driven_theta[idx],
            origin="lower",
            cmap="twilight",
            vmin=-np.pi,
            vmax=np.pi,
        )
        axes[0, col].set_title(f"driven theta, t={driven_time[idx]:.2f}")
        axes[0, col].set_xticks([])
        axes[0, col].set_yticks([])

        im_mag = axes[1, col].imshow(driven_mag[idx], origin="lower", cmap="magma")
        axes[1, col].set_title("driven |z|")
        axes[1, col].set_xticks([])
        axes[1, col].set_yticks([])

    fig.colorbar(im, ax=axes[0, :].ravel().tolist(), fraction=0.025, pad=0.02, label="theta")
    fig.colorbar(im_mag, ax=axes[1, :].ravel().tolist(), fraction=0.025, pad=0.02, label="|z|")
    plt.show()

    row = L // 2
    fig, ax = plt.subplots(figsize=(10, 4.5))
    im = ax.imshow(
        driven_theta[:, row, :],
        aspect="auto",
        origin="lower",
        cmap="twilight",
        vmin=-np.pi,
        vmax=np.pi,
        extent=[0, L, driven_time[0], driven_time[-1]],
    )
    ax.set_xlabel("x")
    ax.set_ylabel("time after burn-in")
    ax.set_title(f"driven space-time angle cut at y={row}")
    fig.colorbar(im, ax=ax, label="theta")
    plt.tight_layout()
    plt.show()

In [ ]:
if RUN_DRIVEN_WAVE:
    max_movie_frames = 240
    fps = 24
    arrow_stride = 4
    SAVE_DRIVEN_MOVIE = True
    DISPLAY_DRIVEN_MOVIE_INLINE = False
    output_dir = Path(CNEEP_V2_ROOT) / "results" / "saou_chiral_sanity"
    output_dir.mkdir(parents=True, exist_ok=True)

    frame_ids = np.linspace(0, len(driven_traj) - 1, min(max_movie_frames, len(driven_traj)), dtype=int)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), dpi=120)
    angle_im = axes[0].imshow(
        driven_theta[frame_ids[0]],
        origin="lower",
        cmap="twilight",
        vmin=-np.pi,
        vmax=np.pi,
    )
    mag_vmax = max(1e-12, float(np.percentile(driven_mag, 99)))
    mag_im = axes[1].imshow(driven_mag[frame_ids[0]], origin="lower", cmap="magma", vmin=0, vmax=mag_vmax)

    ys, xs = np.mgrid[0:L:arrow_stride, 0:L:arrow_stride]
    initial_angle = driven_theta[frame_ids[0], ::arrow_stride, ::arrow_stride]
    quiv = axes[0].quiver(
        xs + 0.5,
        ys + 0.5,
        np.cos(initial_angle),
        np.sin(initial_angle),
        color="black",
        pivot="middle",
        scale=32,
        width=0.003,
        alpha=0.72,
    )

    axes[0].set_title("driven angle theta")
    axes[1].set_title("driven magnitude |z|")
    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
    fig.colorbar(angle_im, ax=axes[0], fraction=0.046, label="theta")
    fig.colorbar(mag_im, ax=axes[1], fraction=0.046, label="|z|")
    title = fig.suptitle("")

    def update_driven_movie(i):
        idx = int(frame_ids[i])
        ang = driven_theta[idx]
        angle_im.set_data(ang)
        mag_im.set_data(driven_mag[idx])
        ang_sub = ang[::arrow_stride, ::arrow_stride]
        quiv.set_UVC(np.cos(ang_sub), np.sin(ang_sub))
        title.set_text(f"Driven SAOU chiral wave | frame={idx}, t={driven_time[idx]:.2f}")
        return angle_im, mag_im, quiv, title

    anim = FuncAnimation(fig, update_driven_movie, frames=len(frame_ids), interval=1000 / fps, blit=False)

    saved_paths = []
    save_errors = []
    mp4_path = output_dir / "saou_driven_chiral_angle_movie.mp4"
    gif_path = output_dir / "saou_driven_chiral_angle_movie.gif"
    html_path = output_dir / "saou_driven_chiral_angle_movie.html"

    if SAVE_DRIVEN_MOVIE:
        for writer_name, path in [("ffmpeg", mp4_path), ("pillow", gif_path)]:
            try:
                if not mpl_animation.writers.is_available(writer_name):
                    save_errors.append(f"{writer_name}: writer unavailable")
                    continue
                if writer_name == "ffmpeg":
                    writer = mpl_animation.FFMpegWriter(fps=fps, bitrate=2400)
                    anim.save(str(path), writer=writer, dpi=120)
                else:
                    writer = mpl_animation.PillowWriter(fps=fps)
                    anim.save(str(path), writer=writer, dpi=100)
                saved_paths.append(path)
                break
            except Exception as exc:
                save_errors.append(f"{writer_name}: {exc}")

        if not saved_paths:
            html_path.write_text(anim.to_jshtml(fps=fps), encoding="utf-8")
            saved_paths.append(html_path)

    html = anim.to_jshtml(fps=fps) if DISPLAY_DRIVEN_MOVIE_INLINE else ""
    plt.close(fig)

    if saved_paths:
        links = "<br>".join(f'<a href="{p.resolve().as_uri()}">{p.name}</a>' for p in saved_paths)
        display(HTML(f"<b>Saved driven animation:</b><br>{links}"))
    if save_errors:
        display(HTML("<br>".join(f"<small>{err}</small>" for err in save_errors)))
    if html:
        display(HTML(html))

## 9. Driven-wave EP-density maps

With a time-dependent drive, the original autonomous SAOU EPR theory no longer applies by itself.  The left map below shows the SAOU irreversible part only; the right map also includes the external drive as a force-work contribution.

In [ ]:
if RUN_DRIVEN_WAVE:
    drive_torch_device = torch.device(device)
    for op in drive_ops:
        op.kernel_hat_t = torch.from_numpy(op.kernel_hat).to(device=drive_torch_device, dtype=torch.complex128)

    def plane_wave_drive_numpy(t):
        phase_t = spatial_phase + omega_mode * float(t)
        force = np.empty((L, L, 2), dtype=np.float64)
        force[..., 0] = drive_amp * np.cos(phase_t)
        force[..., 1] = drive_amp * np.sin(phase_t)
        return force

    def driven_ep_density_between(traj, idx):
        x_start = traj[idx]
        x_end = traj[idx + 1]
        x_mid = 0.5 * (x_start + x_end)
        dx_field = x_end - x_start
        t_mid = 0.5 * (driven_time_abs[idx] + driven_time_abs[idx + 1])
        drive_force_mid = plane_wave_drive_numpy(t_mid)

        with torch.no_grad():
            x_mid_t = torch.from_numpy(x_mid).to(device=drive_torch_device, dtype=torch.float64)
            vel_saou = irreversible_velocity_total(x_mid_t, drive_ops, omega0=omega0).cpu().numpy()

        saou_ep = np.sum(vel_saou * dx_field, axis=-1) / (drive_T * drive_dt_frame)
        drive_power = np.sum(drive_force_mid * dx_field, axis=-1) / (drive_T * drive_dt_frame)
        total_with_drive = saou_ep + drive_power
        return saou_ep, drive_power, total_with_drive

    idx = len(driven_traj) // 2
    saou_ep, drive_power, total_with_drive = driven_ep_density_between(driven_traj, idx)

    vmax_saou = max(1e-12, float(np.percentile(np.abs(saou_ep), 99)))
    vmax_drive = max(1e-12, float(np.percentile(np.abs(drive_power), 99)))
    vmax_total = max(1e-12, float(np.percentile(np.abs(total_with_drive), 99)))

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    im0 = axes[0].imshow(driven_theta[idx], origin="lower", cmap="twilight", vmin=-np.pi, vmax=np.pi)
    axes[0].set_title(f"driven theta, t={driven_time[idx]:.2f}")
    fig.colorbar(im0, ax=axes[0], fraction=0.046)

    im1 = axes[1].imshow(saou_ep, origin="lower", cmap="RdBu_r", vmin=-vmax_saou, vmax=vmax_saou)
    axes[1].set_title("SAOU-only EP rate")
    fig.colorbar(im1, ax=axes[1], fraction=0.046)

    im2 = axes[2].imshow(drive_power, origin="lower", cmap="RdBu_r", vmin=-vmax_drive, vmax=vmax_drive)
    axes[2].set_title("drive work rate")
    fig.colorbar(im2, ax=axes[2], fraction=0.046)

    im3 = axes[3].imshow(total_with_drive, origin="lower", cmap="RdBu_r", vmin=-vmax_total, vmax=vmax_total)
    axes[3].set_title("SAOU + drive rate")
    fig.colorbar(im3, ax=axes[3], fraction=0.046)

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()

    print(f"mean SAOU-only EP rate:     {saou_ep.mean():.6g}")
    print(f"mean drive work rate:       {drive_power.mean():.6g}")
    print(f"mean SAOU + drive rate:     {total_with_drive.mean():.6g}")

## 10. Central pinned rotating spin source

This is the point-source version.  A single central site is softly pinned to a rotating target spin,

`f_center(t) = k_pin [X_target(t) - X_center]`

where

`X_target(t) = A_center [cos(Omega_center t), sin(Omega_center t)]`.

The central spin is therefore an explicit external source.  The autonomous SAOU theory no longer applies globally, but this setup is useful if the goal is to see whether a localized chiral source creates spatial structure in the predicted EP map.  The source work is computed separately from the SAOU-only irreversible contribution.

In [ ]:
RUN_CENTER_SOURCE = True

if RUN_CENTER_SOURCE:
    try:
        from tqdm.auto import trange
    except ImportError:
        trange = range

    center_y = L // 2
    center_x = L // 2
    center_source_mode = "soft_pin"  # "soft_pin" keeps source work measurable; "hard_clamp" overwrites X_center.
    center_source_amplitude = 2.5
    center_pin_strength = 40.0
    center_source_omega = omega_mode
    center_gamma = 0.08
    center_T = 0.006
    center_dt = dt
    center_burn_steps = 10_000
    center_n_steps = 24_000
    center_sample_every = 20
    center_dt_frame = center_dt * center_sample_every

    center_shells = make_annular_shells(
        radii,
        amplitudes,
        weight_normalization=weight_normalization,
    )
    center_ops = build_shell_ops(L, center_shells)
    torch_device = torch.device(device)
    for op in center_ops:
        op.kernel_hat_t = torch.from_numpy(op.kernel_hat).to(device=torch_device, dtype=torch.complex128)

    torch.manual_seed(seed + 40)
    x = 0.03 * torch.randn((L, L, 2), device=torch_device, dtype=torch.float64)
    noise_scale = math.sqrt(2.0 * center_T * center_dt)

    def center_target_torch(t):
        phase = center_source_omega * float(t)
        return torch.tensor(
            [
                center_source_amplitude * math.cos(phase),
                center_source_amplitude * math.sin(phase),
            ],
            device=torch_device,
            dtype=torch.float64,
        )

    def center_pin_force(x_state, t):
        force = torch.zeros_like(x_state)
        target = center_target_torch(t)
        force[center_y, center_x] = center_pin_strength * (target - x_state[center_y, center_x])
        return force

    def maybe_hard_clamp(x_state, t):
        if center_source_mode == "hard_clamp":
            x_state = x_state.clone()
            x_state[center_y, center_x] = center_target_torch(t)
        return x_state

    with torch.no_grad():
        x = maybe_hard_clamp(x, 0.0)
        for step in trange(center_burn_steps, desc="center-source burn-in", leave=False):
            t = step * center_dt
            external = center_pin_force(x, t) if center_source_mode == "soft_pin" else torch.zeros_like(x)
            vel_irr = irreversible_velocity_total(x, center_ops, omega0=omega0)
            x = x + (-center_gamma * x + vel_irr + external) * center_dt
            x = x + noise_scale * torch.randn_like(x)
            x = maybe_hard_clamp(x, (step + 1) * center_dt)

        center_samples = []
        center_sample_times_abs = []
        center_saou_ep_rate_maps = []
        center_source_work_rate_maps = []
        center_ep_acc_saou = torch.zeros((L, L), device=torch_device, dtype=torch.float64)
        center_ep_acc_source = torch.zeros((L, L), device=torch_device, dtype=torch.float64)
        for step in trange(center_n_steps, desc="center-source simulation", leave=False):
            t = (center_burn_steps + step) * center_dt
            x_before = x
            external = center_pin_force(x, t) if center_source_mode == "soft_pin" else torch.zeros_like(x)
            vel_irr = irreversible_velocity_total(x_before, center_ops, omega0=omega0)
            x_after = x_before + (-center_gamma * x_before + vel_irr + external) * center_dt
            x_after = x_after + noise_scale * torch.randn_like(x_after)
            dx_step = x_after - x_before
            center_ep_acc_saou = center_ep_acc_saou + torch.sum(vel_irr * dx_step, dim=-1) / center_T
            center_ep_acc_source = center_ep_acc_source + torch.sum(external * dx_step, dim=-1) / center_T
            x = x_after
            x = maybe_hard_clamp(x, t + center_dt)

            if (step + 1) % center_sample_every == 0:
                center_samples.append(x.cpu().numpy())
                center_sample_times_abs.append((center_burn_steps + step + 1) * center_dt)
                center_saou_ep_rate_maps.append((center_ep_acc_saou / center_dt_frame).cpu().numpy())
                center_source_work_rate_maps.append((center_ep_acc_source / center_dt_frame).cpu().numpy())
                center_ep_acc_saou.zero_()
                center_ep_acc_source.zero_()

    center_traj = np.asarray(center_samples)
    center_time_abs = np.asarray(center_sample_times_abs)
    center_time = center_time_abs - center_burn_steps * center_dt
    center_z = center_traj[..., 0] + 1j * center_traj[..., 1]
    center_theta = np.angle(center_z)
    center_mag = np.abs(center_z)
    center_saou_ep_rate_maps = np.asarray(center_saou_ep_rate_maps)
    center_source_work_rate_maps = np.asarray(center_source_work_rate_maps)
    center_total_ep_rate_maps = center_saou_ep_rate_maps + center_source_work_rate_maps

    print("center-source trajectory:", center_traj.shape)
    print("center-source EP maps:", center_total_ep_rate_maps.shape)
    print(f"source mode:      {center_source_mode}")
    print(f"burn-in time:     {center_burn_steps * center_dt:.3f}")
    print(f"production time:  {center_n_steps * center_dt:.3f}")
    print(f"source amplitude: {center_source_amplitude:.3f}")
    print(f"pin strength:     {center_pin_strength:.3f}")
    print(f"source omega:     {center_source_omega:.6f}")
else:
    print("Center source skipped.")

In [ ]:
if RUN_CENTER_SOURCE:
    center_spin = center_traj[:, center_y, center_x, :]
    center_target = np.stack(
        [
            center_source_amplitude * np.cos(center_source_omega * center_time_abs),
            center_source_amplitude * np.sin(center_source_omega * center_time_abs),
        ],
        axis=-1,
    )
    center_tracking_error = np.linalg.norm(center_spin - center_target, axis=-1)

    yy_grid, xx_grid = np.meshgrid(np.arange(L), np.arange(L), indexing="ij")
    dx_center = ((xx_grid - center_x + L // 2) % L) - L // 2
    dy_center = ((yy_grid - center_y + L // 2) % L) - L // 2
    rr = np.sqrt(dx_center**2 + dy_center**2)
    r_bins = np.arange(0, L // 2 + 1)
    r_centers = 0.5 * (r_bins[:-1] + r_bins[1:])

    radial_mag = np.zeros((len(center_traj), len(r_centers)))
    for b in range(len(r_centers)):
        mask = (rr >= r_bins[b]) & (rr < r_bins[b + 1])
        radial_mag[:, b] = center_mag[:, mask].mean()

    def running_mean_1d(x, window):
        window = max(1, min(int(window), len(x)))
        kernel = np.ones(window, dtype=np.float64) / window
        return np.convolve(x, kernel, mode="same")

    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=False)
    smooth_window = max(5, len(center_time) // 40)

    axes[0].plot(center_time, center_tracking_error, alpha=0.4, lw=0.8)
    axes[0].plot(center_time, running_mean_1d(center_tracking_error, smooth_window), lw=2)
    axes[0].set_ylabel("center tracking error")

    axes[1].plot(center_time, np.sqrt(np.mean(center_mag**2, axis=(1, 2))), alpha=0.4, lw=0.8)
    axes[1].plot(center_time, running_mean_1d(np.sqrt(np.mean(center_mag**2, axis=(1, 2))), smooth_window), lw=2)
    axes[1].set_ylabel("RMS |z|")

    for idx in [0, len(center_time) // 2, len(center_time) - 1]:
        axes[2].plot(r_centers, radial_mag[idx], label=f"t={center_time[idx]:.1f}")
    axes[2].set_xlabel("radius from center")
    axes[2].set_ylabel("radial mean |z|")
    axes[2].legend()

    fig.suptitle("Central source steady diagnostics")
    plt.tight_layout()
    plt.show()

    print(f"tracking error late mean: {center_tracking_error[-len(center_tracking_error)//4:].mean():.5f}")
    print(f"RMS |z| late mean:        {np.sqrt(np.mean(center_mag[-len(center_mag)//4:]**2, axis=(1, 2))).mean():.5f}")

In [ ]:
if RUN_CENTER_SOURCE:
    max_movie_frames = 240
    fps = 24
    arrow_stride = 4
    SAVE_CENTER_MOVIE = True
    DISPLAY_CENTER_MOVIE_INLINE = False
    output_dir = Path(CNEEP_V2_ROOT) / "results" / "saou_chiral_sanity"
    output_dir.mkdir(parents=True, exist_ok=True)

    frame_ids = np.linspace(0, len(center_traj) - 1, min(max_movie_frames, len(center_traj)), dtype=int)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), dpi=120)
    angle_im = axes[0].imshow(
        center_theta[frame_ids[0]],
        origin="lower",
        cmap="twilight",
        vmin=-np.pi,
        vmax=np.pi,
    )
    mag_vmax = max(1e-12, float(np.percentile(center_mag, 99)))
    mag_im = axes[1].imshow(center_mag[frame_ids[0]], origin="lower", cmap="magma", vmin=0, vmax=mag_vmax)

    ys, xs = np.mgrid[0:L:arrow_stride, 0:L:arrow_stride]
    initial_angle = center_theta[frame_ids[0], ::arrow_stride, ::arrow_stride]
    quiv = axes[0].quiver(
        xs + 0.5,
        ys + 0.5,
        np.cos(initial_angle),
        np.sin(initial_angle),
        color="black",
        pivot="middle",
        scale=32,
        width=0.003,
        alpha=0.72,
    )

    for ax in axes:
        ax.scatter([center_x], [center_y], s=90, facecolors="none", edgecolors="white", linewidths=1.6)
        ax.set_xticks([])
        ax.set_yticks([])
    axes[0].set_title("center-source angle theta")
    axes[1].set_title("center-source magnitude |z|")
    fig.colorbar(angle_im, ax=axes[0], fraction=0.046, label="theta")
    fig.colorbar(mag_im, ax=axes[1], fraction=0.046, label="|z|")
    title = fig.suptitle("")

    def update_center_movie(i):
        idx = int(frame_ids[i])
        ang = center_theta[idx]
        angle_im.set_data(ang)
        mag_im.set_data(center_mag[idx])
        ang_sub = ang[::arrow_stride, ::arrow_stride]
        quiv.set_UVC(np.cos(ang_sub), np.sin(ang_sub))
        title.set_text(f"Central rotating source | frame={idx}, t={center_time[idx]:.2f}")
        return angle_im, mag_im, quiv, title

    anim = FuncAnimation(fig, update_center_movie, frames=len(frame_ids), interval=1000 / fps, blit=False)

    saved_paths = []
    save_errors = []
    mp4_path = output_dir / "saou_center_spin_angle_movie.mp4"
    gif_path = output_dir / "saou_center_spin_angle_movie.gif"
    html_path = output_dir / "saou_center_spin_angle_movie.html"

    if SAVE_CENTER_MOVIE:
        for writer_name, path in [("ffmpeg", mp4_path), ("pillow", gif_path)]:
            try:
                if not mpl_animation.writers.is_available(writer_name):
                    save_errors.append(f"{writer_name}: writer unavailable")
                    continue
                if writer_name == "ffmpeg":
                    writer = mpl_animation.FFMpegWriter(fps=fps, bitrate=2400)
                    anim.save(str(path), writer=writer, dpi=120)
                else:
                    writer = mpl_animation.PillowWriter(fps=fps)
                    anim.save(str(path), writer=writer, dpi=100)
                saved_paths.append(path)
                break
            except Exception as exc:
                save_errors.append(f"{writer_name}: {exc}")

        if not saved_paths:
            html_path.write_text(anim.to_jshtml(fps=fps), encoding="utf-8")
            saved_paths.append(html_path)

    html = anim.to_jshtml(fps=fps) if DISPLAY_CENTER_MOVIE_INLINE else ""
    plt.close(fig)

    if saved_paths:
        links = "<br>".join(f'<a href="{p.resolve().as_uri()}">{p.name}</a>' for p in saved_paths)
        display(HTML(f"<b>Saved center-source animation:</b><br>{links}"))
    if save_errors:
        display(HTML("<br>".join(f"<small>{err}</small>" for err in save_errors)))
    if html:
        display(HTML(html))

## 11. Central-source EP-density maps

For `soft_pin`, the source work is estimated as `f_center(x_mid, t_mid) dot dx / T`.  For `hard_clamp`, that work is not well-defined from the overwritten trajectory alone, so the source-work panel is only meaningful in soft-pin mode.

In [ ]:
if RUN_CENTER_SOURCE:
    center_torch_device = torch.device(device)
    for op in center_ops:
        op.kernel_hat_t = torch.from_numpy(op.kernel_hat).to(device=center_torch_device, dtype=torch.complex128)

    idx = len(center_traj) // 2
    center_saou_ep = center_saou_ep_rate_maps[idx]
    center_source_work = center_source_work_rate_maps[idx]
    center_total_ep = center_total_ep_rate_maps[idx]

    vmax_saou = max(1e-12, float(np.percentile(np.abs(center_saou_ep), 99)))
    # Source work is intentionally a one-site sparse signal, so percentile scaling
    # can erase it.  Use max scaling for this panel.
    vmax_source = max(1e-12, float(np.max(np.abs(center_source_work))))
    vmax_total = max(1e-12, float(np.percentile(np.abs(center_total_ep), 99)))

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    im0 = axes[0].imshow(center_theta[idx], origin="lower", cmap="twilight", vmin=-np.pi, vmax=np.pi)
    axes[0].set_title(f"theta, t={center_time[idx]:.2f}")
    fig.colorbar(im0, ax=axes[0], fraction=0.046)

    im1 = axes[1].imshow(center_saou_ep, origin="lower", cmap="RdBu_r", vmin=-vmax_saou, vmax=vmax_saou)
    axes[1].set_title("SAOU-only EP rate")
    fig.colorbar(im1, ax=axes[1], fraction=0.046)

    im2 = axes[2].imshow(center_source_work, origin="lower", cmap="RdBu_r", vmin=-vmax_source, vmax=vmax_source)
    axes[2].set_title("center-source work rate")
    fig.colorbar(im2, ax=axes[2], fraction=0.046)

    im3 = axes[3].imshow(center_total_ep, origin="lower", cmap="RdBu_r", vmin=-vmax_total, vmax=vmax_total)
    axes[3].set_title("SAOU + source rate")
    fig.colorbar(im3, ax=axes[3], fraction=0.046)

    for ax in axes:
        ax.scatter([center_x], [center_y], s=90, facecolors="none", edgecolors="white", linewidths=1.6)
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()

    radial_saou = []
    radial_source = []
    radial_total = []
    for b in range(len(r_centers)):
        mask = (rr >= r_bins[b]) & (rr < r_bins[b + 1])
        radial_saou.append(center_saou_ep[mask].mean())
        radial_source.append(center_source_work[mask].mean())
        radial_total.append(center_total_ep[mask].mean())

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(r_centers, radial_saou, label="SAOU-only")
    ax.plot(r_centers, radial_source, label="source work")
    ax.plot(r_centers, radial_total, label="SAOU + source")
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel("radius from center")
    ax.set_ylabel("ring-mean EP rate")
    ax.set_title("Central-source radial EP profile")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"mean SAOU-only EP rate: {center_saou_ep.mean():.6g}")
    print(f"mean source work rate:  {center_source_work.mean():.6g}")
    print(f"mean total rate:        {center_total_ep.mean():.6g}")
    print(f"center source rate:     {center_source_work[center_y, center_x]:.6g}")

## 12. Quick stationary EPR check

This final cell is a small stationary sanity check.  It is separate from the coherent-mode movie because the movie starts from a deliberately organized initial condition.

In [ ]:
RUN_STATIONARY_EPR_CHECK = True

if RUN_STATIONARY_EPR_CHECK:
    epr_out = simulate(
        L=24,
        M=1,
        device=device,
        radii=radii,
        amplitudes=amplitudes,
        gamma=1.0,
        omega0=omega0,
        T=1.0,
        dt=1.0e-3,
        n_steps=8_000,
        burn_steps=2_000,
        sample_every=20,
        seed=seed + 100,
        record_trajectory=False,
        weight_normalization=weight_normalization,
        show_progress=True,
    )

    gram = epr_out["epr_rate_theory_gram"]
    labels = ["local"] + [sh.name for sh in epr_out["shells"]]
    theory_components = gram.sum(axis=1)
    estimate_components = np.concatenate(
        [[epr_out["epr_rate_est_local"]], epr_out["epr_rate_est_by_shell"]]
    )

    print("Pathwise EPR component rates: estimated vs theory")
    print("-" * 72)
    for label, est, th in zip(labels, estimate_components, theory_components):
        print(f"{label:14s}  est={est:12.4f}  theory={th:12.4f}  diff={est - th:12.4f}")
    print("-" * 72)
    print(f"{'total':14s}  est={epr_out['epr_rate_est_total']:12.4f}  theory={epr_out['epr_rate_theory_total']:12.4f}")
else:
    print("Skipped stationary EPR check.")